In [6]:
# --- BLOQUE 0: INFRAESTRUCTURA ---
import sys
import subprocess
import warnings

warnings.filterwarnings('ignore')

def install(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", package])

print("[SYSTEM] Asegurando que las bibliotecas científicas estén actualizadas...")
install("astroquery")
install("astropy")
install("scipy")
install("matplotlib")

from astroquery.heasarc import Heasarc
import astropy
import scipy
import matplotlib

import numpy as np
import matplotlib.pyplot as plt
from astropy.cosmology import Planck18 as cosmo
from scipy.optimize import curve_fit

print("[SYSTEM] Iniciando Protocolo ASTRO-MINER v6 (Hardcoded Columns)...")

# --- BLOQUE 1: CONSULTA SQL (Target Confirmado) ---
heasarc = Heasarc()
TARGET_TABLE = 'fermilgrb'

print(f"[QUERY] Descargando tabla {TARGET_TABLE} completa...")

try:
    # Pedimos la tabla completa usando el método correcto para misiones
    table = heasarc.query_table(TARGET_TABLE)
    df = table.to_pandas()
    # Normalizar columnas a mayúsculas para evitar errores tontos
    df.columns = [c.upper() for c in df.columns]
    print(f"[SUCCESS] Tabla recuperada. Registros: {len(df)}")

except Exception as e:
    print(f"[CRITICAL] Error de conexión HEASARC: {e}")
    sys.exit()

# --- BLOQUE 2: SELECCIÓN DE COLUMNAS (QUIRÚRGICO) ---
# Usamos las columnas confirmadas en tu log anterior.
# Z -> REDSHIFT
# E -> EXT_EMISSION_MAX_ENE (MeV)
# T -> EXT_EMISSION_MAX_ENE_T (s)

print("[MAP] Mapeando columnas confirmadas...")

try:
    # Copia de seguridad de datos crudos
    raw_z = df['REDSHIFT']
    raw_e = df['EXT_EMISSION_MAX_ENE']     # MeV
    raw_t = df['EXT_EMISSION_MAX_ENE_T']   # s (desde el trigger)
    raw_n = df['NAME']

except KeyError as e:
    print(f"[ERROR] La columna {e} no existe (A pesar de aparecer en el log).")
    print("Intenta usar 'GBM_CAT_T90' o similar si 'EXT_EMISSION' falla.")
    sys.exit()

# --- BLOQUE 3: FILTRADO Y FÍSICA ---
# Convertimos MeV a GeV
raw_e_gev = raw_e / 1000.0

# Filtros de Calidad Física
# 1. Redshift válido (> 0)
# 2. Energía significativa (> 1 GeV) - Bajamos un poco el umbral para tener más puntos
# 3. Tiempo válido (no NaN)
mask = (raw_z > 0) & (np.isfinite(raw_z)) & (raw_e_gev > 1.0) & (np.isfinite(raw_t))

clean_z = raw_z[mask].values
clean_t = raw_t[mask].values
clean_e = raw_e_gev[mask].values
clean_names = raw_n[mask].values

print(f"[DATA] Eventos útiles para LIV (z>0, E>1GeV): {len(clean_z)}")

if len(clean_z) < 3:
    print("[WARN] Muy pocos datos. Intentando recuperar eventos sin emisión extendida...")
    # Fallback: Usar T90 y Energía máxima de verosimilitud si EXT falla
    # Pero por ahora, veamos el gráfico con lo que hay.
    pass

# Cálculo K-Factor
print("[PHYS] Integrando cosmología...")
def get_k(z):
    z_grid = np.linspace(0, z, 50)
    Ez = np.sqrt(cosmo.Om0*(1+z_grid)**3 + cosmo.Ode0)
    integrand = (1+z_grid) / Ez
    return np.trapz(integrand, z_grid)

K_factors = np.array([get_k(z) for z in clean_z])

# Observable: Retraso por GeV
lags = clean_t / clean_e

# --- BLOQUE 4: ANÁLISIS ---
# Regresión
def model(k, m, c):
    return m * k + c

# Ajuste robusto (L1 o simple least squares)
if len(K_factors) > 2:
    popt, pcov = curve_fit(model, K_factors, lags)
    m_fit = popt[0]
    m_err = np.sqrt(np.diag(pcov))[0]
else:
    m_fit, m_err = 0, 0
    print("[SKIP] No hay suficientes puntos para ajustar una recta.")

# --- BLOQUE 5: VISUALIZACIÓN ---
plt.figure(figsize=(12, 7))
plt.style.use('dark_background')

plt.errorbar(K_factors, lags, yerr=0.1*np.abs(lags)+0.1, fmt='o', color='#00ffcc',
             ecolor='gray', alpha=0.8, label='Datos Fermi-LAT (Ext. Emission)')

if len(K_factors) > 2:
    x_plot = np.linspace(0, max(K_factors)*1.1, 50)
    plt.plot(x_plot, model(x_plot, m_fit, popt[1]), color='#ff00ff',
             linestyle='--', label=f'Ajuste LIV (m={m_fit:.2e})')

plt.axhline(0, color='white', linestyle=':', label='Einstein (m=0)')

# Etiquetas
for i, name in enumerate(clean_names):
    if clean_e[i] > 10.0 or abs(lags[i]) > 10: # Etiquetar los extremos
        plt.annotate(f"{name}\n{clean_e[i]:.1f}GeV",
                     (K_factors[i], lags[i]), color='yellow', fontsize=8)

plt.xlabel('Factor K (Distancia)')
plt.ylabel('Retraso Normalizado (s/GeV)')
plt.title('Cazando Violación de Lorentz (v6)\nDatos: HEASARC fermilgrb [EXT_EMISSION]', color='white')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\n[RESULT] Pendiente m = {m_fit:.4e} +/- {m_err:.4e}")

[SYSTEM] Asegurando que las bibliotecas científicas estén actualizadas...


ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



[SYSTEM] Iniciando Protocolo ASTRO-MINER v6 (Hardcoded Columns)...
[QUERY] Descargando tabla fermilgrb completa...
[CRITICAL] Error de conexión HEASARC: 'HeasarcClass' object has no attribute 'query_mission_catalog'
Traceback (most recent call last):
  File "/tmp/ipython-input-296635709.py", line 37, in <cell line: 0>
    table = heasarc.query_mission_catalog(mission=TARGET_TABLE)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AttributeError: 'HeasarcClass' object has no attribute 'query_mission_catalog'. Did you mean: 'query_mission_cols'?

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipython-input-296635709.py", line 45, in <cell line: 0>
    sys.exit()
SystemExit

During handling of the above exception, another exception occurred:

Traceback (most recent call 

TypeError: object of type 'NoneType' has no len()